In [1]:
# Check GPU
!nvidia-smi

# Install ultralytics (YOLOv8)
!pip install ultralytics

# (Optional) Install YOLOv11 if available in ultralytics-nightly
!pip install git+https://github.com/ultralytics/ultralytics.git@main

Sat Jan  3 04:42:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
import ultralytics
from ultralytics import YOLO

print("PyTorch version:", torch.__version__)
print("Ultralytics version:", ultralytics.__version__)

PyTorch version: 2.9.0+cu126
Ultralytics version: 8.3.246


In [3]:
# Example: Roboflow dataset download
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key=YOUR_API_KEY)   # get API key from Roboflow account
project = rf.workspace("trial-7qqqy").project("car-parts-elf2f-7w8oe")
dataset = project.version("1").download("yolov8")   # replace x with version number

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to CAR-parts-1 in yolov8:: 100%|██████████| 2008/2008 [00:00<00:00, 5686.70it/s]


In [4]:
import os

# Path to your dataset folder (adjust if needed)
dataset_path = "/content/CAR-parts-1"

# Count images in train/valid/test splits
for split in ["train", "valid", "test"]:
    img_dir = os.path.join(dataset_path, split, "images")
    if os.path.exists(img_dir):
        print(split, ":", len(os.listdir(img_dir)), "images")

train : 699 images
valid : 199 images
test : 100 images


In [5]:
# Train YOLOv8 segmentation model
model = YOLO("yolov8n-seg.pt")  # you can use yolov8s-seg.pt, yolov8m-seg.pt, etc.

model.train(
    data="/content/CAR-parts-1/data.yaml",
    epochs=150,
    imgsz=640,
    batch=16,
    device=0
)


Ultralytics 8.3.246 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/CAR-parts-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, po

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7ebe7d3d9010>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.03503

In [7]:
!zip -r runs_car_parts.zip /content/runs/segment/train

  adding: content/runs/segment/train/ (stored 0%)
  adding: content/runs/segment/train/labels.jpg (deflated 20%)
  adding: content/runs/segment/train/val_batch2_labels.jpg (deflated 5%)
  adding: content/runs/segment/train/val_batch0_pred.jpg (deflated 5%)
  adding: content/runs/segment/train/train_batch0.jpg (deflated 3%)
  adding: content/runs/segment/train/val_batch0_labels.jpg (deflated 6%)
  adding: content/runs/segment/train/MaskF1_curve.png (deflated 10%)
  adding: content/runs/segment/train/BoxP_curve.png (deflated 10%)
  adding: content/runs/segment/train/args.yaml (deflated 52%)
  adding: content/runs/segment/train/val_batch1_labels.jpg (deflated 6%)
  adding: content/runs/segment/train/train_batch6161.jpg (deflated 8%)
  adding: content/runs/segment/train/weights/ (stored 0%)
  adding: content/runs/segment/train/weights/last.pt (deflated 9%)
  adding: content/runs/segment/train/weights/best.pt (deflated 9%)
  adding: content/runs/segment/train/results.csv (deflated 63%)
  ad